# Fine-tune CRAFT on REAL Hebrew handwriting (weak-supervised)

Trains CRAFT's region/affinity heatmaps on **real word crops from dataset_matan** with
pseudo per-letter boxes (connected components, kept only where the CC count matched the GT
letter count). This targets real handwriting, not synthetic fonts.

**Setup:** run `python src/prep_craft_data.py` locally → upload `craft_train.zip` to Drive
→ paste its link below. Internet ON, GPU ON, Run All. Output: `craft_hebrew.pth`.

In [ ]:
# ── 0. CRAFT code + weights + real-data package ────────────────────────
!pip install -q gdown
import os, re, sys, glob, json, zipfile, pathlib
!git clone -q https://github.com/clovaai/CRAFT-pytorch
p = pathlib.Path('CRAFT-pytorch/basenet/vgg16_bn.py'); s = p.read_text()
s = s.replace('from torchvision.models.vgg import model_urls', '')
s = s.replace("model_urls['vgg16_bn'] = model_urls['vgg16_bn'].replace('https://', 'http://')", '')
s = s.replace('models.vgg16_bn(pretrained=pretrained)', 'models.vgg16_bn(weights=None)')
p.write_text(s); sys.path.insert(0, 'CRAFT-pytorch')
!curl -fsSL -o craft_mlt_25k.pth https://huggingface.co/boomb0om/dataset-filters/resolve/main/craft_mlt_25k.pth
import gdown
TRAIN_URL = 'https://drive.google.com/file/d/1lQ7ig-Vu2ikdW1E0ns36-MCmEbuI-jAn/view?usp=sharing'
fid = next(g for g in re.search(r'/d/([\w-]+)|[?&]id=([\w-]+)', TRAIN_URL).groups() if g)
gdown.download(f'https://drive.google.com/uc?id={fid}', 'craft_train.zip', quiet=False)
zipfile.ZipFile('craft_train.zip').extractall('.')
BOXES = json.load(open('craft_train/boxes.json'))
IMGS = sorted(glob.glob('craft_train/imgs/*.png'))
print('training word crops:', len(IMGS))


In [ ]:
# ── 1. params + heatmap targets from boxes ─────────────────────────────
import numpy as np, cv2, torch
TH, TW = 64, 256; OUT_H, OUT_W = TH//2, TW//2     # canvas (matches prep) + CRAFT half-res output
SEED=1234; rng=np.random.default_rng(SEED); EPOCHS, BATCH, LR = 12, 16, 1e-4
def gaussian_template(size=64):
    ax=np.linspace(-1,1,size); xx,yy=np.meshgrid(ax,ax)
    g=np.exp(-(xx**2+yy**2)/(2*0.35**2)); return (g/g.max()).astype(np.float32)
TMPL=gaussian_template()
def place(heat,box):
    x0,y0,x1,y1=[int(v) for v in box]; w,h=max(x1-x0,1),max(y1-y0,1)
    g=cv2.resize(TMPL,(w,h)); H,W=heat.shape
    x0=max(x0,0);y0=max(y0,0);x1=min(x0+w,W);y1=min(y0+h,H)
    if x1<=x0 or y1<=y0: return
    heat[y0:y1,x0:x1]=np.maximum(heat[y0:y1,x0:x1],g[:y1-y0,:x1-x0])
def targets(boxes):
    reg=np.zeros((OUT_H,OUT_W),np.float32); aff=np.zeros((OUT_H,OUT_W),np.float32)
    hb=[(x0/2,y0/2,x1/2,y1/2) for (x0,y0,x1,y1) in boxes]
    for b in hb: place(reg,b)
    sb=sorted(hb,key=lambda b:b[0])
    for a,b in zip(sb,sb[1:]):
        cy=((a[1]+a[3])/2+(b[1]+b[3])/2)/2; bh=((a[3]-a[1])+(b[3]-b[1]))/4
        place(aff,[(a[0]+a[2])/2,cy-bh,(b[0]+b[2])/2,cy+bh])
    return reg,aff
import imgproc
def load_sample(fn):
    g=cv2.imread(fn,cv2.IMREAD_GRAYSCALE)
    if rng.random()<0.4: g=cv2.GaussianBlur(g,(0,0),rng.uniform(0.3,0.9))   # light aug
    reg,aff=targets(BOXES[os.path.basename(fn)])
    x=imgproc.normalizeMeanVariance(cv2.cvtColor(g,cv2.COLOR_GRAY2RGB))
    return x.transpose(2,0,1), reg, aff
def make_batch(files):
    xs,rs,as_=zip(*[load_sample(f) for f in files])
    return (torch.from_numpy(np.stack(xs)).float(),
            torch.from_numpy(np.stack(rs)), torch.from_numpy(np.stack(as_)))


In [ ]:
# ── 2. load CRAFT, init from craft_mlt_25k ─────────────────────────────
from collections import OrderedDict
from craft import CRAFT
def copy_sd(sd): return OrderedDict((k[7:],v) for k,v in sd.items()) if list(sd)[0].startswith('module') else sd
dev='cuda' if torch.cuda.is_available() else 'cpu'
net=CRAFT(); net.load_state_dict(copy_sd(torch.load('craft_mlt_25k.pth',map_location='cpu'))); net=net.to(dev).train()
print('CRAFT on',dev)


In [ ]:
# ── 3. fine-tune (MSE on region + affinity) ────────────────────────────
from tqdm.auto import trange
opt=torch.optim.Adam(net.parameters(),lr=LR); mse=torch.nn.MSELoss()
files=list(IMGS); best=1e9
for ep in range(EPOCHS):
    rng.shuffle(files); tot=0.0; nb=0
    for i in trange(0,len(files)-BATCH,BATCH,desc=f'epoch {ep+1}/{EPOCHS}',leave=False):
        x,rg,af=make_batch(files[i:i+BATCH]); x=x.to(dev); rg=rg.to(dev); af=af.to(dev)
        y,_=net(x); loss=mse(y[:,:,:,0],rg)+mse(y[:,:,:,1],af)
        opt.zero_grad(); loss.backward(); opt.step(); tot+=loss.item(); nb+=1
    avg=tot/max(nb,1); print(f'epoch {ep+1} loss={avg:.4f}')
    if avg<best: best=avg; torch.save(net.state_dict(),'craft_hebrew.pth')
print('best loss',best)


In [ ]:
# ── 4. sanity: predicted region on a training sample ───────────────────
import matplotlib.pyplot as plt
net.eval(); fn=IMGS[0]; g=cv2.imread(fn,cv2.IMREAD_GRAYSCALE)
x=torch.from_numpy(imgproc.normalizeMeanVariance(cv2.cvtColor(g,cv2.COLOR_GRAY2RGB)).transpose(2,0,1))[None].float().to(dev)
with torch.no_grad(): y,_=net(x)
reg=y[0,:,:,0].cpu().numpy(); rg,_=targets(BOXES[os.path.basename(fn)])
fig,ax=plt.subplots(3,1,figsize=(8,4))
ax[0].imshow(g,cmap='gray');ax[0].set_title('input word');ax[0].axis('off')
ax[1].imshow(rg,cmap='jet');ax[1].set_title('target region');ax[1].axis('off')
ax[2].imshow(reg,cmap='jet');ax[2].set_title('predicted region');ax[2].axis('off')
plt.tight_layout(); plt.show()


In [ ]:
# ── 5. download craft_hebrew.pth ───────────────────────────────────────
import os
from IPython.display import FileLink, display
os.chdir('/kaggle/working'); print('MB:', round(os.path.getsize('craft_hebrew.pth')/1e6,1)); display(FileLink('craft_hebrew.pth'))
